In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings

warnings.filterwarnings('ignore')

In [4]:
user_inputs_example = {
    'Area': 150.0,
    'Crop_Rice': 1,
    'State_Uttar Pradesh': 1,
    'Season_Kharif': 1
}

In [5]:
def load_model_and_columns():
    """
    Loads the pre-trained model and the list of feature columns.
    Assumes 'random_forest_model.joblib' and 'model_features.joblib' are present.
    """
    try:
        model = joblib.load('random_forest_model.joblib')
        columns = joblib.load('model_features.joblib')
        return model, columns
    except FileNotFoundError as e:
        print(f"Error loading model files: {e}. Please ensure you have run the training notebook first.")
        return None, None

def get_automated_data():
    """
    This function simulates fetching data from a weather or agricultural API.
    In a real-world scenario, this would make an API call based on the
    user's location, crop type, and time of year.
    For this example, we'll return fixed values.
    """
    return {
        'Annual_Rainfall': 1200.0,
        'Fertilizer': 500.0,
        'Pesticide': 150.0,
        'Crop_Year': 2020,
    }

In [14]:
def analyze_and_recommend(user_inputs):
    """
    Analyzes the user's inputs and provides a recommendation on which factor
    to adjust to increase the crop yield.
    """
    model, loaded_columns = load_model_and_columns()
    if not model or not loaded_columns:
        return "Cannot proceed with analysis. Model files are missing."

    # Get automated data and combine with user inputs
    automated_data = get_automated_data()
    input_data = {**user_inputs, **automated_data}

    base_df = pd.DataFrame([input_data])
    for col in loaded_columns:
        if col not in base_df.columns:
            base_df[col] = 0
    base_df = base_df[loaded_columns]

    # Get the initial prediction
    initial_yield = model.predict(base_df)[0]
    print(f"Initial Predicted Yield for your inputs: {initial_yield:.2f}")


    # Filter for numerical features that can be controlled (not natural factors)
    controllable_features = ['Fertilizer', 'Pesticide']
    
    recommendations = {}
    
    # Simulate adjustments to each controllable feature
    for feature in controllable_features:
        # Simulate a 10% increase
        simulated_input_increase = base_df.copy()
        simulated_input_increase[feature] *= 1.10
        predicted_yield_increase = model.predict(simulated_input_increase)[0]
        recommendations[f"Increase {feature}"] = predicted_yield_increase

        # Simulate a 10% decrease
        simulated_input_decrease = base_df.copy()
        simulated_input_decrease[feature] *= 0.90
        predicted_yield_decrease = model.predict(simulated_input_decrease)[0]
        recommendations[f"Decrease {feature}"] = predicted_yield_decrease

    # Sort the recommendations to find the top performers
    sorted_recommendations = sorted(recommendations.items(), key=lambda item: item[1], reverse=True)
    
    # Check if there are any positive recommendations
    if sorted_recommendations[0][1] <= initial_yield:
        return "Based on the current conditions, adjustments to these factors may not significantly improve the yield. Consider other variables or consult local experts."
    
    # Build the final recommendation message with the top 3 suggestions
    message = "\n--- Yield Optimization Recommendations ---\n"
    message += "Here are the top actions you can take to increase your yield:\n\n"
    
    for i, (action, predicted_yield) in enumerate(sorted_recommendations):
        # Only show recommendations that actually increase the yield
        if predicted_yield > initial_yield:
            yield_increase_percent = ((predicted_yield - initial_yield) / initial_yield) * 100
            message += f"{i+1}. **{action}**\n"
            message += f"   - Potential Yield: **{predicted_yield:.2f}**\n"
            message += f"   - Improvement: **{yield_increase_percent:.2f}%**\n"
    
    return message

In [15]:
print("\n--- Running Optimization Analysis ---")
recommendation_text = analyze_and_recommend(user_inputs_example)
print(recommendation_text)


--- Running Optimization Analysis ---
Initial Predicted Yield for your inputs: 2.56

--- Yield Optimization Recommendations ---
Here are the top actions you can take to increase your yield:

1. **Increase Pesticide**
   - Potential Yield: **2.57**
   - Improvement: **0.48%**

